# EDA — Sample Superstore Dataset

Análise exploratória de dados (EDA) da base de vendas **Sample Superstore** (Kaggle), para estudo da estrutura, tipos de dados e inconsistências antes da modelagem no Power BI.

Ajuste o caminho do arquivo na célula abaixo antes de rodar.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## Carregar os dados

In [2]:
CAMINHO_ARQUIVO = "Sample - Superstore.csv"  

def carregar_dados(caminho: str) -> pd.DataFrame:
    """Carrega o CSV tentando encodings comuns em exports do Kaggle."""
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(caminho, encoding=enc)
            print(f"[OK] Arquivo lido com encoding '{enc}'.")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError("Não foi possível ler o arquivo com os encodings testados.")

df = carregar_dados(CAMINHO_ARQUIVO)

[OK] Arquivo lido com encoding 'latin-1'.


## 0. Checagem inicial — arquivo vazio

Confirma que o arquivo não veio só com cabeçalho (problema real que já aconteceu numa exportação anterior do Kaggle).

In [3]:
linhas_totalmente_vazias = df.isna().all(axis=1).sum()

if linhas_totalmente_vazias == len(df) and len(df) > 0:
    print("[ALERTA] O arquivo contém apenas o cabeçalho — todas as linhas estão vazias. "
          "Verifique a exportação de origem (Kaggle/Excel/Google Sheets) antes de prosseguir.")
else:
    print(f"[OK] Arquivo com dados: {len(df)} linhas preenchidas.")

[OK] Arquivo com dados: 9994 linhas preenchidas.


## 1. Estrutura geral

In [4]:
print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
print("\nTipos de dados por coluna:")
print(df.dtypes)

Linhas: 9994 | Colunas: 21

Tipos de dados por coluna:
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object


In [5]:
df.head().T

,0,1,2,3,4
Row ID,1,2,3,4,5
Order ID,CA-2016-152156,CA-2016-152156,CA-2016-138688,US-2015-108966,US-2015-108966
Order Date,11/8/2016,11/8/2016,6/12/2016,10/11/2015,10/11/2015
Ship Date,11/11/2016,11/11/2016,6/16/2016,10/18/2015,10/18/2015
Ship Mode,Second Class,Second Class,Second Class,Standard Class,Standard Class
Customer ID,CG-12520,CG-12520,DV-13045,SO-20335,SO-20335
Customer Name,Claire Gute,Claire Gute,Darrin Van Huff,Sean O'Donnell,Sean O'Donnell
Segment,Consumer,Consumer,Corporate,Consumer,Consumer
Country,United States,United States,United States,United States,United States
City,Henderson,Henderson,Los Angeles,Fort Lauderdale,Fort Lauderdale


## 2. Valores ausentes

In [6]:
nulos = df.isna().sum()
pct = (nulos / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({"nulos": nulos, "% do total": pct})
resumo_nulos = resumo_nulos[resumo_nulos["nulos"] > 0]

if resumo_nulos.empty:
    print("Nenhum valor ausente encontrado em nenhuma coluna.")
else:
    display(resumo_nulos)

Nenhum valor ausente encontrado em nenhuma coluna.


## 3. Linhas duplicadas

In [7]:
dup = df.duplicated().sum()
print(f"Linhas 100% duplicadas: {dup}")

if "Row ID" in df.columns:
    dup_id = df["Row ID"].duplicated().sum()
    print(f"'Row ID' duplicado: {dup_id}")

Linhas 100% duplicadas: 0
'Row ID' duplicado: 0


## 4. Consistência de categorias

In [8]:
colunas_categoricas = ["Ship Mode", "Segment", "Region", "Category", "Sub-Category"]

for col in colunas_categoricas:
    if col not in df.columns:
        continue
    valores = sorted(df[col].dropna().unique().tolist())
    print(f"\n{col} ({len(valores)} valores únicos):")
    print(valores if len(valores) <= 20 else valores[:20] + ["..."])


Ship Mode (4 valores únicos):
['First Class', 'Same Day', 'Second Class', 'Standard Class']

Segment (3 valores únicos):
['Consumer', 'Corporate', 'Home Office']

Region (4 valores únicos):
['Central', 'East', 'South', 'West']

Category (3 valores únicos):
['Furniture', 'Office Supplies', 'Technology']

Sub-Category (17 valores únicos):
['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']


## 5. Consistência de datas

In [9]:
for col in ["Order Date", "Ship Date"]:
    if col not in df.columns:
        continue
    datas = pd.to_datetime(df[col], format="%m/%d/%Y", errors="coerce")
    invalidas = datas.isna().sum() - df[col].isna().sum()
    print(f"{col}: {datas.min().date()} a {datas.max().date()} "
          f"| datas inválidas (não convertidas): {invalidas}")

if "Order Date" in df.columns and "Ship Date" in df.columns:
    od = pd.to_datetime(df["Order Date"], format="%m/%d/%Y", errors="coerce")
    sd = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y", errors="coerce")
    inconsistentes = (sd < od).sum()
    print(f"\nPedidos com Data de Envio anterior à Data do Pedido: {inconsistentes}")

Order Date: 2014-01-03 a 2017-12-30 | datas inválidas (não convertidas): 0


Ship Date: 2014-01-07 a 2018-01-05 | datas inválidas (não convertidas): 0

Pedidos com Data de Envio anterior à Data do Pedido: 0


## 6. Consistência de valores numéricos

In [10]:
checagens = {
    "Sales": lambda s: (s < 0).sum(),
    "Quantity": lambda s: (s <= 0).sum(),
    "Discount": lambda s: ((s < 0) | (s > 1)).sum(),
    "Profit": lambda s: None,  # profit pode ser negativo legitimamente (prejuízo)
}

for col, teste in checagens.items():
    if col not in df.columns:
        continue
    resultado = teste(df[col]) if teste else None
    print(f"{col}: min={df[col].min()}, max={df[col].max()}, média={df[col].mean():.2f}"
          + (f" | valores fora do esperado: {resultado}" if resultado is not None else ""))

Sales: min=0.444, max=22638.48, média=229.86 | valores fora do esperado: 0
Quantity: min=1, max=14, média=3.79 | valores fora do esperado: 0
Discount: min=0.0, max=0.8, média=0.16 | valores fora do esperado: 0
Profit: min=-6599.978, max=8399.976, média=28.66


## 7. Estatísticas descritivas

In [11]:
df.describe().round(2)

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00
mean,4997.50,55190.38,229.86,3.79,0.16,28.66
std,2885.16,32063.69,623.25,2.23,0.21,234.26
min,1.00,1040.00,0.44,1.00,0.00,-6599.98
25%,2499.25,23223.00,17.28,2.00,0.00,1.73
50%,4997.50,56430.50,54.49,3.00,0.20,8.67
75%,7495.75,90008.00,209.94,5.00,0.20,29.36
max,9994.00,99301.00,22638.48,14.00,0.80,8399.98


## 8. Resumo de negócio

In [12]:
if "Sales" in df.columns:
    total_vendas = df["Sales"].sum()
    print(f"Faturamento total: US$ {total_vendas:,.2f}")
if "Order ID" in df.columns:
    pedidos_unicos = df["Order ID"].nunique()
    print(f"Pedidos únicos: {pedidos_unicos}")

Faturamento total: US$ 2,297,200.86
Pedidos únicos: 5009


In [13]:
if "Category" in df.columns and "Sales" in df.columns:
    print("Faturamento por categoria:")
    display(df.groupby("Category")["Sales"].sum().sort_values(ascending=False).round(2))

Faturamento por categoria:


Category
Technology         836154.03
Furniture          741999.80
Office Supplies    719047.03
Name: Sales, dtype: float64

In [14]:
if "Region" in df.columns and "Sales" in df.columns:
    print("Faturamento por região:")
    display(df.groupby("Region")["Sales"].sum().sort_values(ascending=False).round(2))

Faturamento por região:


Region
West       725457.82
East       678781.24
Central    501239.89
South      391721.90
Name: Sales, dtype: float64